In [1]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE, URL = '/content/code', 'github.com/quocthangtrann/computer_vision_ADAS.git'
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
if not os.path.exists(CODE):
    subprocess.run(f'git clone https://{URL} {CODE}', shell=True, check=True, env=env)
else:
    subprocess.run(f'git -C {CODE} pull', shell=True, check=True, env=env)
subprocess.run(f'rsync -a --exclude .git {CODE}/repo/ /content/drive/MyDrive/capstone_ai/repo/', shell=True, check=True)
try: import segmentation_models_pytorch
except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch'], check=True)
os.chdir('/content/drive/MyDrive/capstone_ai')
print('mã mới nhất đã ở capstone_ai/repo/ | thư mục làm việc:', os.getcwd())


Mounted at /content/drive
mã mới nhất đã ở capstone_ai/repo/ | thư mục làm việc: /content/drive/MyDrive/capstone_ai


In [3]:
import os, subprocess
print(subprocess.run('git -C /content/code log --oneline -1', shell=True, capture_output=True, text=True).stdout.strip())
for f in ['repo/make_hd_pilot.py', 'repo/train_v3.py', 'repo/models_custom.py']:
    print(f, 'có' if os.path.exists(f) else 'THIẾU')
for f in ['data/bdd100k_seg.tar', 'data/v2/drv_masks_3274.tar', 'data/v2/pilot_drv_x_320x160.npy', 'data/v2/pilot_drv_y_320x160.npy',
          'data/v2/pilot_drv_manifest.csv', 'data/v2/pilot_drv_x_640x320.npy', 'data/v2/pilot_drv_y_640x320.npy']:
    print(f, f'{round(os.path.getsize(f) / 1e6)} MB' if os.path.exists(f) else 'KHÔNG CÓ')

708a40d Huấn luyện HD: giáo viên transformer, chưng cất, chống quá khớp, đọc dữ liệu từ đĩa, notebook 70_hd
repo/make_hd_pilot.py có
repo/train_v3.py có
repo/models_custom.py có
data/bdd100k_seg.tar 1329 MB
data/v2/drv_masks_3274.tar 13 MB
data/v2/pilot_drv_x_320x160.npy 497 MB
data/v2/pilot_drv_y_320x160.npy 166 MB
data/v2/pilot_drv_manifest.csv 0 MB
data/v2/pilot_drv_x_640x320.npy KHÔNG CÓ
data/v2/pilot_drv_y_640x320.npy KHÔNG CÓ


In [4]:
import os, shutil
os.chdir('/content/drive/MyDrive/capstone_ai')
os.makedirs('data/v2/backup_pilot_320', exist_ok=True)
for f in ['pilot_drv_x_320x160.npy', 'pilot_drv_y_320x160.npy', 'pilot_drv_manifest.csv']:
    dst = f'data/v2/backup_pilot_320/{f}'
    if not os.path.exists(dst): shutil.copyfile(f'data/v2/{f}', dst)
    print(f, f'{os.path.getsize(dst) / 1e6:.0f} MB', '| trùng kích thước với bản gốc:', os.path.getsize(dst) == os.path.getsize(f'data/v2/{f}'))

pilot_drv_x_320x160.npy 497 MB | trùng kích thước với bản gốc: True
pilot_drv_y_320x160.npy 166 MB | trùng kích thước với bản gốc: True
pilot_drv_manifest.csv 0 MB | trùng kích thước với bản gốc: True


In [5]:
# CẤU HÌNH: chạy cell 0 và cell này ở MỖI phiên Colab (CPU hay GPU đều vậy)
W, H = 640, 320     # 1280, 576 chỉ nên dùng với NS = 0 (xem hướng dẫn: shard 1280x576 nặng khoảng 22 GB mỗi shard)
NS = 0            # số shard b9 (0 = chỉ bộ thí điểm; 1 so với b9s1_c4aug_s*; 4 so với b9s4_c4aug_s*)
TAG = f'hd{W}s{NS}'
SHARDS = ','.join(f'shard_{i:03d}' for i in range(NS))
BASE = {0: 'pilot_c4aug_s*', 1: 'b9s1_c4aug_s*', 4: 'b9s4_c4aug_s*'}.get(NS)   # mốc 320x160 cùng dữ liệu và cùng aug
print(f'cỡ {W}x{H} | {NS} shard ({SHARDS or "không"}) | tag {TAG} | mốc so sánh: {BASE}')


cỡ 640x320 | 0 shard (không) | tag hd640s0 | mốc so sánh: pilot_c4aug_s*


In [7]:
import os, subprocess, sys
# Chạy được trên runtime CPU. Kiểm chứng trước: dựng lại 320x160 và so từng pixel với cache có sẵn, chỉ dựng cỡ HD khi ĐẠT.
if os.path.exists(f'data/v2/pilot_drv_x_{W}x{H}.npy') and os.path.exists(f'data/v2/pilot_drv_y_{W}x{H}.npy'):
    print(f'đã có cache thí điểm {W}x{H}, bỏ qua (xóa hai tệp pilot_drv_*_{W}x{H}.npy trên Drive nếu muốn dựng lại)')
else:
    for w, h in [(320, 160), (W, H)]:
        r = subprocess.run([sys.executable, '-u', 'repo/make_hd_pilot.py', '--w', str(w), '--h', str(h)])
        if r.returncode != 0: raise SystemExit(f'make_hd_pilot {w}x{h} lỗi (mã {r.returncode}): dừng, dán đoạn trên cho tôi')


In [8]:
import numpy as np
os.chdir('/content/drive/MyDrive/capstone_ai')
for f in ['pilot_drv_x_320x160.npy', 'pilot_drv_y_320x160.npy']:
    a = np.load(f'data/v2/{f}', mmap_mode='r'); b = np.load(f'data/v2/backup_pilot_320/{f}', mmap_mode='r')
    print(f, 'cache 320x160 còn nguyên:', a.shape == b.shape and all(np.array_equal(a[i:i + 256], b[i:i + 256]) for i in range(0, len(a), 256)))
for f in ['pilot_drv_x_640x320.npy', 'pilot_drv_y_640x320.npy']:
    a = np.load(f'data/v2/{f}', mmap_mode='r'); print(f, a.shape, a.dtype, f'{os.path.getsize(f"data/v2/{f}") / 1e6:.0f} MB')
y = np.load('data/v2/pilot_drv_y_640x320.npy', mmap_mode='r'); v = np.unique(np.asarray(y[:200]))
print('giá trị nhãn trong 200 ảnh đầu:', v.tolist(), '(mong đợi: tập con của 0, 1, 2, 3, 255)')

pilot_drv_x_320x160.npy cache 320x160 còn nguyên: True
pilot_drv_y_320x160.npy cache 320x160 còn nguyên: True
pilot_drv_x_640x320.npy (3234, 320, 640, 3) uint8 1987 MB
pilot_drv_y_640x320.npy (3234, 320, 640) uint8 662 MB
giá trị nhãn trong 200 ảnh đầu: [0, 1, 2, 3, 255] (mong đợi: tập con của 0, 1, 2, 3, 255)


In [9]:
import os, io, shutil, tarfile, numpy as np, pandas as pd
from PIL import Image
os.chdir('/content/drive/MyDrive/capstone_ai')
W, H = 640, 320
os.makedirs('/content/raw', exist_ok=True)
LOC = '/content/raw/bdd100k_seg.tar'
if not os.path.exists(LOC): shutil.copy('data/bdd100k_seg.tar', LOC)
tf = tarfile.open(LOC); mem = {m.name: m for m in tf.getmembers() if m.isfile()}
J = {os.path.basename(n)[:-4]: n for n in mem if n.endswith('.jpg') and '/images_10k/' in n}
S = {os.path.basename(n)[:-13]: n for n in mem if n.endswith('_train_id.png')}
dt = tarfile.open('data/v2/drv_masks_3274.tar'); D = {os.path.basename(m.name)[:-4]: m for m in dt.getmembers() if m.isfile()}
man = pd.read_csv('data/v2/pilot_drv_manifest.csv')
X = np.load(f'data/v2/pilot_drv_x_{W}x{H}.npy', mmap_mode='r'); Y = np.load(f'data/v2/pilot_drv_y_{W}x{H}.npy', mmap_mode='r')
assert len(man) == len(X) == len(Y), (len(man), len(X), len(Y))
CROP = (0, 0, 1280, 600)
idx = np.sort(np.random.default_rng(0).choice(len(man), 24, replace=False)); bx = by = 0
for k in idx:
    i = man.id[k]
    im = Image.open(io.BytesIO(tf.extractfile(mem[J[i]]).read())).convert('RGB')
    s = np.asarray(Image.open(io.BytesIO(tf.extractfile(mem[S[i]]).read())).crop(CROP).resize((W, H), Image.NEAREST))
    d = np.asarray(Image.open(io.BytesIO(dt.extractfile(D[i]).read())).crop(CROP).resize((W, H), Image.NEAREST))
    road, ign = s == 0, s == 255
    lab = road.astype(np.uint8); lab[d == 1] = 2; lab[d == 2] = 3
    lab[((d == 1) | (d == 2)) & ~road & ~ign] = 255; lab[ign] = 255
    bx += int((np.asarray(im.crop(CROP).resize((W, H), Image.BILINEAR)) != np.asarray(X[k])).sum()); by += int((lab != np.asarray(Y[k])).sum())
print(f'{len(idx)} ảnh | pixel ảnh lệch: {bx} | pixel nhãn lệch: {by}')
print('ĐẠT' if bx == 0 and by == 0 else 'LỆCH: dừng, dán kết quả cho tôi')

24 ảnh | pixel ảnh lệch: 0 | pixel nhãn lệch: 0
ĐẠT


In [ ]:
import os, subprocess, sys
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# Cần cache thí điểm cùng cỡ ở cell trên. Shard đã có trên Drive được bỏ qua; có thể chạy lại sau khi bị ngắt.
if NS == 0: print('NS = 0: không cần shard')
else:
    os.makedirs('runs', exist_ok=True)
    with open(f'runs/b9_prepare_{W}x{H}.log', 'a') as lg:
        pr = subprocess.Popen([sys.executable, '-u', 'repo/b9_prepare_v2.py', '--shards', str(NS), '--w', str(W), '--h', str(H)],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for ln in pr.stdout: lg.write(ln); print(ln, end='', flush=True)
    if pr.wait() != 0: raise SystemExit('b9_prepare_v2 lỗi: dán đoạn log trên cho tôi')


In [10]:
import sys
sys.path.insert(0, 'repo')
import models_custom as mc
import segmentation_models_pytorch as smp
bad_all = {}
for n in ['fsnet_m_p1', 'fsnet_m_p1s5', 'fsnet_l_p1', 'fsnet_l_p1s5', 'fsnet_m', 'fsnet_l']:
    pc = mc.PRESETS[n]; need = 2 ** (4 + pc.get('pre', 0) + (1 if pc.get('e5') else 0))
    if W % need or H % need: print(f'{n:14s} {W}x{H}: bỏ qua (cần w, h chia hết cho {need})'); continue
    net = mc.build_fsnet(n, num_classes=4).eval(); rows = mc.layer_table(net, H, W)
    ho, wo = mc.fuse_for_deploy(net)(__import__('torch').zeros(1, 3, H, W)).shape[2:]
    bad_all[n] = mc.contract_violations(net)
    print(f'{n:14s} {W}x{H}  {sum(r["mac"] for r in rows) / 1e9:7.3f} GMAC | {sum(p.numel() for p in net.parameters()) / 1e6:.3f} M tham số | lưới logit {wo}x{ho} | vi phạm hợp đồng: {bad_all[n] or "không"}')
assert not any(bad_all.values()), 'có mô hình vi phạm hợp đồng toán tử'
for e in ['mit_b2', 'resnet34']:
    m = smp.Unet(e, encoder_weights=None, classes=4)
    print(f'giáo viên smp:Unet:{e}: encoder {sum(p.numel() for p in m.encoder.parameters()) / 1e6:.2f} M | tổng {sum(p.numel() for p in m.parameters()) / 1e6:.2f} M tham số')


fsnet_m_p1     640x320    0.651 GMAC | 0.726 M tham số | lưới logit 80x40 | vi phạm hợp đồng: không
fsnet_m_p1s5   640x320    0.725 GMAC | 1.316 M tham số | lưới logit 80x40 | vi phạm hợp đồng: không
fsnet_l_p1     640x320    1.158 GMAC | 1.478 M tham số | lưới logit 80x40 | vi phạm hợp đồng: không
fsnet_l_p1s5   640x320    1.324 GMAC | 2.806 M tham số | lưới logit 80x40 | vi phạm hợp đồng: không
fsnet_m        640x320    2.339 GMAC | 0.726 M tham số | lưới logit 160x80 | vi phạm hợp đồng: không
fsnet_l        640x320    4.367 GMAC | 1.478 M tham số | lưới logit 160x80 | vi phạm hợp đồng: không
giáo viên smp:Unet:mit_b2: encoder 24.20 M | tổng 27.48 M tham số
giáo viên smp:Unet:resnet34: encoder 21.28 M | tổng 24.44 M tham số


In [13]:
import subprocess, sys, time, os
import torch; assert torch.cuda.is_available(), 'Chưa có GPU: đổi runtime sang GPU'
ONLY = 's'       # None = cả hàng đợi; hoặc chuỗi nhóm: 't' giáo viên, 's' học trò thường, 'k' học trò chưng cất, 'l' học trò lớn
EPOCHS_T, EPOCHS_S = 40, 60
BIG = W >= 1280   # cỡ 1280x576: lô nhỏ hơn, cộng dồn gradient nhiều hơn (chưa đo bộ nhớ: nếu hết bộ nhớ thì giảm --bs, tăng --accum)
SZ = ['--w', str(W), '--h', str(H)]
B9 = ['--b9', SHARDS] if NS else []
T_BS = ['--bs', '2', '--accum', '8'] if BIG else ['--bs', '8', '--accum', '2']
S_BS = ['--bs', '8', '--accum', '4'] if BIG else []
T_OPT = ['--lr', '6e-4', '--lr_enc_mult', '0.1', '--wd', '0.01', '--wd_nobn', '1', '--ema', '0.999']   # encoder tiền huấn luyện: lr 6e-5
REG = ['--ema', '0.999', '--wd', '0.01', '--wd_nobn', '1', '--ls', '0.05', '--cutout', '2']              # gói chống quá khớp cho học trò
KD = ['--kd', '1.0', '--kd_T', '2.0']                                                                     # trọng số KD 1,0: CHƯA chỉnh
TM, TC = f'{TAG}_mitb2_c4aug_s0', f'{TAG}_unetr34_c4aug_s0'
# (nhóm, tag, tham số thêm cho train_v3.py, các seed, số epoch, giáo viên cần có). Tên run = {tag}_c4aug_s{seed}
JOBS = [
    ('t', f'{TAG}_mitb2',    ['--arch', 'smp:Unet:mit_b2'] + T_BS + T_OPT, [0], EPOCHS_T, None),
    ('t', f'{TAG}_unetr34', ['--arch', 'smp:Unet:resnet34'] + T_BS + T_OPT, [0], EPOCHS_T, None),
    ('s', f'{TAG}_p1s5',     ['--arch', 'fsnet_m_p1s5'] + S_BS, [0, 1, 2], EPOCHS_S, None),
    ('s', f'{TAG}_p1s5_reg', ['--arch', 'fsnet_m_p1s5'] + S_BS + REG, [0, 1, 2], EPOCHS_S, None),
    ('k', f'{TAG}_p1s5_kdT', ['--arch', 'fsnet_m_p1s5'] + S_BS + REG + KD + ['--teacher', TM], [0, 1, 2], EPOCHS_S, TM),
    ('k', f'{TAG}_p1s5_kdC', ['--arch', 'fsnet_m_p1s5'] + S_BS + REG + KD + ['--teacher', TC], [0, 1, 2], EPOCHS_S, TC),
    ('l', f'{TAG}_lp1s5_kdT', ['--arch', 'fsnet_l_p1s5'] + S_BS + REG + KD + ['--teacher', TM], [0, 1, 2], EPOCHS_S, TM),
]

JOBS = [(g, t, e, ([0] if g == 's' else sd), ep, th) for g, t, e, sd, ep, th in JOBS]

def run_queue():
    T = time.time()
    for grp, tag, extra, seeds, epochs, teacher in JOBS:
        if ONLY and grp not in ONLY: continue
        for s in seeds:
            n = f'{tag}_c4aug_s{s}'
            if os.path.exists(f'runs/{n}/final_metrics.json'): print(n, 'đã xong, bỏ qua'); continue
            if teacher and not os.path.exists(f'runs/{teacher}/final_metrics.json'):
                print(f'DỪNG: {n} cần giáo viên {teacher} đã huấn luyện xong (chạy nhóm t trước)'); return False
            print(f'\n>>> [{grp}] {n}  [{(time.time() - T) / 60:.1f} phút]', flush=True)
            cmd = [sys.executable, '-u', 'repo/train_v3.py', '--name', n, '--seed', str(s), '--epochs', str(epochs)] + SZ + B9 + extra
            pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            with open(f'runs/{n}.log', 'a') as lg:
                for ln in pr.stdout:
                    lg.write(ln)
                    if not ln.startswith('ep ') or int(ln.split()[1].split('/')[0]) in (1, ) or int(ln.split()[1].split('/')[0]) % 10 == 0: print(ln, end='', flush=True)
            rc = pr.wait(); print('mã thoát:', rc, flush=True)
            if rc != 0: print(f'LỖI ở {n}: dừng hàng đợi, dán đoạn log trên cho tôi'); return False
    print(f'\nXONG sau {(time.time() - T) / 60:.1f} phút'); return True
run_queue()



>>> [s] hd640s0_p1s5_c4aug_s0  [0.0 phút]
train 2826 ảnh (thí điểm 2826 + b9 0) | cỡ 640x320 | test 408 | pixel nhãn 254 (ước lượng): 0.0% | nạp hết lên GPU
ảnh nhãn đủ: 2826/2826 (100.0%) | full_w 1.0: tỉ lệ mẫu nhãn đủ mỗi epoch 100.0% | tập đo khoảng cách train: 408 ảnh
mô hình fsnet_m_p1s5: 1.32 M tham số
ep 1/60 loss 0.9871 IoU [0.892, 0.136, 0.484, 0.019] road 0.7118 | IoU320 [0.893, 0.136, 0.485, 0.019] road320 0.7118 25s
ep 10/60 loss 0.4440 IoU [0.943, 0.354, 0.669, 0.425] road 0.8299 | IoU320 [0.943, 0.353, 0.667, 0.425] road320 0.8297 | train 0.695 - test 0.669 (làn đang đi) 10s
ep 20/60 loss 0.3660 IoU [0.949, 0.407, 0.733, 0.526] road 0.8445 | IoU320 [0.949, 0.404, 0.731, 0.526] road320 0.8438 | train 0.764 - test 0.733 (làn đang đi) 10s
ep 30/60 loss 0.3273 IoU [0.955, 0.438, 0.759, 0.555] road 0.8653 | IoU320 [0.955, 0.438, 0.76, 0.555] road320 0.8658 | train 0.796 - test 0.759 (làn đang đi) 10s
ep 40/60 loss 0.3046 IoU [0.958, 0.434, 0.758, 0.572] road 0.8729 | IoU320 

True

In [ ]:
import json, glob, numpy as np
def load(pat): return [json.load(open(p)) for p in sorted(glob.glob(f'runs/{pat}/final_metrics.json'))]
def metrics(f):   # run HD: dùng iou_320 (quy về lưới 320x160) để so với mốc; run 320x160 không có thì dùng iou
    k = '_320' if 'iou_320' in f else ''
    i = f['iou' + k]
    return dict(ego=i[2], adj=i[3], notours=i[1], road=f['road_iou' + k], ff=f['false_floor' + k])
def col(fs, c): return np.array([metrics(f)[c] for f in fs])
def welch(a, b):
    d = b.mean() - a.mean()
    if len(a) < 2 or len(b) < 2: return d, float('nan')
    return d, d / np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
def report(title, base_pat, pat):
    A, B = load(base_pat), load(pat)
    if not B or not A: print(f'== {title}: chưa đủ run (nhóm {len(B)}, mốc {len(A)})\n'); return
    print(f'== {title}: n = {len(B)} so với {base_pat} (n = {len(A)})')
    for c in ['ego', 'adj', 'notours', 'road', 'ff']:
        a, b = col(A, c), col(B, c)
        d, t = welch(a, b)
        sa = f'{a.std(ddof=1):.4f}' if len(a) > 1 else ' n=1  '
        sb = f'{b.std(ddof=1):.4f}' if len(b) > 1 else ' n=1  '
        ts = 'n/a' if t != t else f'{t:+.1f}'
        z = f' | không có t, hiệu / độ lệch chuẩn mốc = {d / a.std(ddof=1):+.1f}' if (len(b) < 2 and len(a) > 1) else ''
        print(f'  {c:8s} mốc {a.mean():.4f} ± {sa} | nhóm {b.mean():.4f} ± {sb} | hiệu {d:+.4f} (t = {ts}){z}')
    print()
R = lambda tag: f'{TAG}_{tag}_c4aug_s*'
if BASE is None: print('NS không có mốc 320x160 tương ứng: chỉ so các nhóm với nhau')
print('######## 1) GIÁO VIÊN = trần (so với mốc FSNet 320x160 cùng dữ liệu). Nếu làn đang đi hơn mốc dưới 0,01 thì chưng cất khó giúp.')
for t in ['mitb2', 'unetr34']:
    if BASE: report(f'giáo viên {t}', BASE, R(t))
print('######## 2) HỌC TRÒ HD so với mốc 320x160')
if BASE:
    for t in ['p1s5', 'p1s5_reg', 'p1s5_kdT', 'p1s5_kdC', 'lp1s5_kdT']: report(f'{t} so với mốc', BASE, R(t))
print('######## 3) TÁCH TỪNG YẾU TỐ (cùng dữ liệu, cùng cỡ)')
report('chống quá khớp (reg) so với không reg', R('p1s5'), R('p1s5_reg'))
report('KD từ transformer so với reg không KD', R('p1s5_reg'), R('p1s5_kdT'))
report('KD từ conv so với reg không KD', R('p1s5_reg'), R('p1s5_kdC'))
report('KD transformer so với KD conv (đối chứng)', R('p1s5_kdC'), R('p1s5_kdT'))
report('học trò lớn (fsnet_l_p1s5) so với nhỏ, cùng KD transformer', R('p1s5_kdT'), R('lp1s5_kdT'))
print('######## 4) KHOẢNG CÁCH QUÁ KHỚP: làn đang đi, ảnh train nhãn đủ (không aug) trừ test, cùng lưới của run')
for t in ['p1s5', 'p1s5_reg', 'p1s5_kdT', 'p1s5_kdC', 'lp1s5_kdT', 'mitb2', 'unetr34']:
    fs = load(R(t))
    if fs:
        tr = np.array([f['iou_trainsub'][2] for f in fs]); te = np.array([f['iou'][2] for f in fs])
        print(f'  {t:12s} train {tr.mean():.4f} - test {te.mean():.4f} = {tr.mean() - te.mean():+.4f} (n = {len(fs)})')


In [14]:
import os, subprocess, glob, torch
os.chdir('/content/drive/MyDrive/capstone_ai')
print('GPU:', torch.cuda.get_device_name(0), '| bộ nhớ', round(torch.cuda.get_device_properties(0).total_memory / 1e9), 'GB')
print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout.strip())
print('--- train_v3.py: nhánh smp và trọng số tiền huấn luyện ---')
print(subprocess.run('grep -n "encoder_weights\\|imagenet\\|smp:" repo/train_v3.py', shell=True, capture_output=True, text=True).stdout)
import segmentation_models_pytorch as smp
smp.Unet('mit_b2', encoder_weights='imagenet', classes=4); print('OK: tải được trọng số ImageNet cho MiT-B2')
print('thư mục shard HD đang có:', glob.glob('data/v2/b9_chunks*'))
print('run HD đã có:', sorted(os.path.basename(p) for p in glob.glob('runs/hd640*')))

GPU: NVIDIA L4 | bộ nhớ 24 GB
overlay         236G   11G  226G   5% /
--- train_v3.py: nhánh smp và trọng số tiền huấn luyện ---
3:Mô hình: FSNet (--arch fsnet_*) hoặc giáo viên smp (--arch smp:Unet:mit_b2, smp:Unet:mobilenet_v2; cần w, h chia hết cho 32).
29:p.add_argument("--pretrained", type=int, default=1, help="arch smp:*: nạp trọng số ImageNet cho encoder")
43:SMP = a.arch.startswith("smp:")
166:    if arch.startswith("smp:"):
168:        return getattr(smp, dec)(enc, encoder_weights="imagenet" if pretrained else None, classes=C)



config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 98.9MB            

model.safetensors: downloading bytes:           |  0.00B            

OK: tải được trọng số ImageNet cho MiT-B2
thư mục shard HD đang có: ['data/v2/b9_chunks']
run HD đã có: ['hd640s0_p1s5_c4aug_s0', 'hd640s0_p1s5_c4aug_s0.log', 'hd640s0_p1s5_reg_c4aug_s0', 'hd640s0_p1s5_reg_c4aug_s0.log']


In [15]:
import os, sys, time, glob, re, subprocess
import numpy as np, torch
os.chdir('/content/drive/MyDrive/capstone_ai'); os.makedirs('runs', exist_ok=True)
assert torch.cuda.is_available(), 'Chưa có GPU'
try:
    from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as ex: print('không đọc được HF_TOKEN:', ex)

W, H = 640, 320
BUDGET_MIN = 400      # không bắt đầu lần chạy mới sau mức này (phút kể từ lúc chạy cell)
GATE_T_MIN = 100      # teacher ước tính lâu hơn mức này thì dừng và chạy bản 20 epoch
T0 = time.time()
T_BS = ['--bs', '8', '--accum', '2']
T_OPT = ['--lr', '6e-4', '--lr_enc_mult', '0.1', '--wd', '0.01', '--wd_nobn', '1', '--ema', '0.999']
REG = ['--ema', '0.999', '--wd', '0.01', '--wd_nobn', '1', '--ls', '0.05', '--cutout', '2']
KD = ['--kd', '1.0', '--kd_T', '2.0']
STU = ['--arch', 'fsnet_m_p1s5']
SH = lambda n: ['--b9', ','.join(f'shard_{i:03d}' for i in range(n))] if n else []

def ready():
    out = []
    for p in glob.glob(f'data/v2/b9_chunks*{W}x{H}*/shard_*_ids.csv'):
        s, d = os.path.basename(p)[:-8], os.path.dirname(p)
        if os.path.exists(f'{d}/{s}_x.npy') and os.path.exists(f'{d}/{s}_y.npy'): out.append(s)
    return sorted(out)

dl = None
def start_download(n_shards=2):
    global dl
    if dl is None or dl.poll() is not None:
        dl = subprocess.Popen([sys.executable, '-u', 'repo/b9_prepare_v2.py', '--shards', str(n_shards), '--w', str(W), '--h', str(H)],
                              stdout=open(f'runs/b9_prepare_{W}x{H}.log', 'a'), stderr=subprocess.STDOUT)
        print(f'đã chạy nền tải shard HD (log: runs/b9_prepare_{W}x{H}.log)', flush=True)

def run_job(name, extra, seed, epochs, gate_min=None):
    if os.path.exists(f'runs/{name}/final_metrics.json'): print(name, 'đã xong, bỏ qua'); return True
    cmd = [sys.executable, '-u', 'repo/train_v3.py', '--name', name, '--seed', str(seed), '--epochs', str(epochs), '--w', str(W), '--h', str(H)] + extra
    print(f'\n>>> {name} [{(time.time() - T0) / 60:.0f} phút]', flush=True)
    pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True); secs = []
    with open(f'runs/{name}.log', 'a') as lg:
        for ln in pr.stdout:
            lg.write(ln); m = re.match(r'ep (\d+)/(\d+)', ln)
            if not m: print(ln, end='', flush=True); continue
            e = int(m.group(1)); t = re.search(r'(\d+)s\s*$', ln)
            if t and e >= 2: secs.append(int(t.group(1)))
            if e in (1, 2) or e % 10 == 0: print(ln, end='', flush=True)
            if gate_min and e == 3 and secs:
                proj = np.mean(secs) * epochs / 60
                print(f'  [cổng thời gian] {np.mean(secs):.0f} s/epoch -> ước tính {proj:.0f} phút cho {epochs} epoch', flush=True)
                if proj > gate_min: pr.kill(); pr.wait(); print('  VƯỢT CỔNG: dừng bản này'); return 'slow'
    rc = pr.wait(); print('mã thoát:', rc, flush=True); return rc == 0

start_download(2)
TAG0 = f'hd{W}s0'
TM, TM20 = f'{TAG0}_mitb2_c4aug_s0', f'{TAG0}_mitb2e20_c4aug_s0'
tx = ['--arch', 'smp:Unet:mit_b2'] + T_BS + T_OPT
if os.path.exists(f'runs/{TM20}/final_metrics.json'): TM, r = TM20, True
else:
    r = run_job(TM, tx, 0, 40, gate_min=GATE_T_MIN)
    if r == 'slow': TM = TM20; r = run_job(TM, tx, 0, 20)
assert r is True, 'teacher lỗi: dừng, dán đoạn log trên cho tôi'
print('teacher dùng:', TM)

FILL = [(f'{TAG0}_p1s5_reg_c4aug_s{s}', STU + REG, s) for s in (1, 2)]
waited = 0
while len(ready()) < 1:
    if dl.poll() is not None and not ready(): raise SystemExit(f'tải shard HD đã dừng mà chưa có shard nào: xem runs/b9_prepare_{W}x{H}.log')
    job = next((j for j in FILL if not os.path.exists(f'runs/{j[0]}/final_metrics.json')), None)
    if job and (time.time() - T0) / 60 < BUDGET_MIN:
        assert run_job(job[0], job[1], job[2], 60) is True, 'lỗi ở lần chạy đệm'
    else:
        time.sleep(120); waited += 2
        if waited % 10 == 0: print(f'đang chờ shard HD... {waited} phút', flush=True)

n = len(ready()); TAGN = f'hd{W}s{n}'; print('shard HD sẵn sàng:', ready(), '| tag', TAGN, flush=True)
OFF = [(f'{TAGN}_p1s5_kdT_c4aug_s0', STU + REG + KD + ['--teacher', TM] + SH(n), 0),
       (f'{TAGN}_p1s5_reg_c4aug_s0', STU + REG + SH(n), 0),
       (f'{TAGN}_p1s5_kdT_c4aug_s1', STU + REG + KD + ['--teacher', TM] + SH(n), 1),
       (f'{TAGN}_p1s5_kdT_c4aug_s2', STU + REG + KD + ['--teacher', TM] + SH(n), 2)]
for nm, ex, sd in OFF:
    if (time.time() - T0) / 60 > BUDGET_MIN: print('hết ngân sách thời gian: dừng'); break
    if run_job(nm, ex, sd, 60) is not True: break
print(f'\nXONG sau {(time.time() - T0) / 60:.0f} phút | shard hiện có: {ready()}' + (' | đã có thêm shard: chạy lại cell này để chạy vòng NS mới' if len(ready()) > n else ''))

đã chạy nền tải shard HD (log: runs/b9_prepare_640x320.log)

>>> hd640s0_mitb2_c4aug_s0 [0 phút]
train 2826 ảnh (thí điểm 2826 + b9 0) | cỡ 640x320 | test 408 | pixel nhãn 254 (ước lượng): 0.0% | nạp hết lên GPU
ảnh nhãn đủ: 2826/2826 (100.0%) | full_w 1.0: tỉ lệ mẫu nhãn đủ mỗi epoch 100.0% | tập đo khoảng cách train: 408 ảnh
mô hình smp:Unet:mit_b2: 27.48 M tham số
nhóm tham số (lr, weight_decay, số tensor): [(5.9999999999999995e-05, 0.01, 113), (5.9999999999999995e-05, 0.0, 219), (0.0006, 0.01, 11), (0.0006, 0.0, 21)]
ep 1/40 loss 1.1052 IoU [0.952, 0.321, 0.66, 0.304] road 0.8567 | IoU320 [0.953, 0.32, 0.661, 0.303] road320 0.8570 91s
ep 2/40 loss 0.4630 IoU [0.962, 0.426, 0.742, 0.504] road 0.8825 | IoU320 [0.962, 0.426, 0.742, 0.505] road320 0.8824 70s
  [cổng thời gian] 70 s/epoch -> ước tính 47 phút cho 40 epoch
ep 10/40 loss 0.2579 IoU [0.969, 0.513, 0.808, 0.652] road 0.9065 | IoU320 [0.969, 0.511, 0.807, 0.652] road320 0.9058 | train 0.875 - test 0.808 (làn đang đi) 73s
ep 2

In [16]:
import os, json, glob, numpy as np, torch
os.chdir('/content/drive/MyDrive/capstone_ai')
def L(pat):
    r = []
    for p in sorted(glob.glob(f'runs/{pat}/final_metrics.json')):
        f = json.load(open(p)); i = f.get('iou_320', f['iou']); r.append((i[2], i[3], i[1], f.get('road_iou_320', f['road_iou'])))
    return np.array(r)
G = {'b9s4 320x160': 'b9s4_c4aug_s*', 'HD kdT 1 shard': 'hd640s1_p1s5_kdT_c4aug_s*', 'HD reg 1 shard': 'hd640s1_p1s5_reg_c4aug_s*'}
D = {k: L(v) for k, v in G.items()}
for k, a in D.items():
    print(f'{k:16s} n={len(a)} | ' + ' | '.join(f'{a[:, j].mean():.4f}' + (f' ± {a[:, j].std(ddof=1):.4f}' if len(a) > 1 else '') for j in range(4)))
a, b = D['b9s4 320x160'], D['HD kdT 1 shard']
for j, nm in enumerate(['làn đang đi', 'làn kề', 'không thuộc ta', 'IoU đường']):
    d = b[:, j].mean() - a[:, j].mean(); se = np.sqrt(a[:, j].var(ddof=1) / len(a) + b[:, j].var(ddof=1) / len(b))
    print(f'{nm:15s} HD kdT trừ b9s4: {d:+.4f} (t = {d / se:+.1f})')
p = 'runs/hd640s1_p1s5_kdT_c4aug_s0'
ck = torch.load(f'{p}/last.pt', map_location='cpu', weights_only=False)
print('\nkhóa trong last.pt:', list(ck.keys()), '| epoch', ck.get('epoch'))
print(json.load(open(f'{p}/config.json')))
print(os.popen('grep -n "ema" repo/train_v3.py | head -30').read())

b9s4 320x160     n=3 | 0.8032 ± 0.0028 | 0.6244 ± 0.0038 | 0.4562 ± 0.0036 | 0.8700 ± 0.0004
HD kdT 1 shard   n=3 | 0.8100 ± 0.0041 | 0.6484 ± 0.0066 | 0.5045 ± 0.0021 | 0.8927 ± 0.0017
HD reg 1 shard   n=1 | 0.8151 | 0.6553 | 0.4854 | 0.8798
làn đang đi     HD kdT trừ b9s4: +0.0068 (t = +2.4)
làn kề          HD kdT trừ b9s4: +0.0240 (t = +5.5)
không thuộc ta  HD kdT trừ b9s4: +0.0484 (t = +19.8)
IoU đường       HD kdT trừ b9s4: +0.0227 (t = +22.4)

khóa trong last.pt: ['model', 'opt', 'sched', 'scaler', 'epoch', 'raw'] | epoch 60
{'name': 'hd640s1_p1s5_kdT_c4aug_s0', 'arch': 'fsnet_m_p1s5', 'epochs': 60, 'bs': 32, 'lr': 0.001, 'dice': 0.5, 'seed': 0, 'aug': 'strong', 'b9': 'shard_000', 'w': 640, 'h': 320, 'full_w': 1.0, 'pretrained': 1, 'teacher': 'hd640s0_mitb2_c4aug_s0', 'kd': 1.0, 'kd_T': 2.0, 'ema': 0.999, 'wd': 0.01, 'wd_nobn': 1, 'lr_enc_mult': 1.0, 'ls': 0.05, 'cutout': 2, 'accum': 1, 'stream': -1, 'gap_n': 408}
8:  --ema --wd --wd_nobn --ls --cutout   chống quá khớp (xem READM

In [17]:
import os, json, glob, shutil, hashlib, zipfile, numpy as np, torch
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/capstone_ai')
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
os.makedirs('models_final', exist_ok=True); os.makedirs('results', exist_ok=True)

# 1) đóng băng 3 mô hình: chỉ lấy ck["model"] (trọng số dùng để đánh giá và triển khai, bản EMA nếu có)
FINAL = {'A_b9s4_fsnet_m_320x160_s0': 'runs/b9s4_c4aug_s0',
         'B_hd640_p1s5_kdT_s0': 'runs/hd640s1_p1s5_kdT_c4aug_s0',
         'teacher_mitb2_hd640': 'runs/hd640s0_mitb2_c4aug_s0'}
info = {}
for name, d in FINAL.items():
    ck = torch.load(f'{d}/last.pt', map_location='cpu', weights_only=False)
    out = f'models_final/{name}_weights.pt'; torch.save(ck['model'], out)
    for f in ['config.json', 'final_metrics.json']: shutil.copy(f'{d}/{f}', f'models_final/{name}_{f}')
    info[name] = dict(run=d, epoch=int(ck.get('epoch', -1)), bytes=os.path.getsize(out), sha256=sha(out))
    print(f'{name}: epoch {info[name]["epoch"]} | {info[name]["bytes"] / 1e6:.1f} MB | sha256 {info[name]["sha256"][:16]}...')
json.dump(info, open('models_final/MODELS.json', 'w'), indent=1)

# 2) bảng kết quả cuối (epoch cuối, lưới 320x160, 408 ảnh test)
def L(pat):
    r = []
    for p in sorted(glob.glob(f'runs/{pat}/final_metrics.json')):
        f = json.load(open(p)); i = f.get('iou_320', f['iou']); r.append((i[2], i[3], i[1], f.get('road_iou_320', f['road_iou'])))
    return np.array(r)
G = [('Mốc thí điểm 2.826 ảnh, aug mạnh, 320x160', 'pilot_c4aug_s*'), ('+1 shard, 320x160', 'b9s1_c4aug_s*'),
     ('A: +4 shard, 320x160 (đã có INT8)', 'b9s4_c4aug_s*'), ('HD 640x320 p1s5_reg, thí điểm', 'hd640s0_p1s5_reg_c4aug_s*'),
     ('Teacher MiT-B2, HD, thí điểm', 'hd640s0_mitb2_c4aug_s0'), ('HD 640x320 p1s5_reg, +1 shard', 'hd640s1_p1s5_reg_c4aug_s*'),
     ('B: HD 640x320 p1s5 + chưng cất, +1 shard', 'hd640s1_p1s5_kdT_c4aug_s*')]
lines, res = ['| Nhóm | n | Làn đang đi | Làn kề | Không thuộc ta | IoU đường |', '|---|---|---|---|---|---|'], {}
for title, pat in G:
    a = L(pat)
    if not len(a): continue
    cells = [f'{a[:, j].mean():.4f}' + (f' ± {a[:, j].std(ddof=1):.4f}' if len(a) > 1 else '') for j in range(4)]
    lines.append(f'| {title} | {len(a)} | ' + ' | '.join(cells) + ' |'); res[title] = dict(n=len(a), mean=a.mean(0).tolist())
md = '# Kết quả huấn luyện (epoch cuối, lưới 320x160, 408 ảnh test, GPU L4)\n\n' + '\n'.join(lines) + '\n'
open('results/final_results.md', 'w', encoding='utf-8').write(md); json.dump(res, open('results/final_results.json', 'w'), indent=1, ensure_ascii=False)
print('\n' + md)

# 3) làm mới gói ngữ cảnh cho Claude Code (chỉ file nhỏ dạng chữ)
pats = ['runs/*/config.json', 'runs/*/final_metrics.json', 'runs/*/diag_last.json', 'runs/*/metrics.csv', 'runs/summary.csv', 'eval_report/*',
        'data_spec_v1.json', 'data/cache_v1/cache_spec.json', 'data/v2/manifest_v2.csv', 'data/v2/split_report.json', 'data/v2/pilot_drv_manifest.csv',
        'handoff_s3a_v2/README.md', 'handoff_s3a_v2/preprocess_spec.json', 'handoff_s3a_v2/layer_table.csv', 'handoff_s3b/README.md',
        'handoff_s3b/int8_meta.json', 'handoff_s3b/int8_sim_summary.json', 'handoff_s3b/export_info.json', 'handoff_s3b/layer_table.csv',
        'handoff_s3b/int8_layers_pct.csv', 'results/*', 'models_final/MODELS.json']
shutil.rmtree('context_pack', ignore_errors=True); os.makedirs('context_pack'); n = 0
for pat in pats:
    for p in glob.glob(pat):
        if os.path.isfile(p) and os.path.getsize(p) < 3_000_000:
            dst = os.path.join('context_pack', p); os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(p, dst); n += 1
with zipfile.ZipFile('context_pack.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for d, _, fs in os.walk('context_pack'):
        for f in fs: z.write(os.path.join(d, f), os.path.relpath(os.path.join(d, f), 'context_pack'))
print(n, 'file | context_pack.zip', round(os.path.getsize('context_pack.zip') / 1e6, 1), 'MB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
A_b9s4_fsnet_m_320x160_s0: epoch 60 | 2.9 MB | sha256 208b69434d4469fa...
B_hd640_p1s5_kdT_s0: epoch 60 | 5.3 MB | sha256 767cf45dbfd2635b...
teacher_mitb2_hd640: epoch 40 | 110.1 MB | sha256 94a66984f0650fc6...

# Kết quả huấn luyện (epoch cuối, lưới 320x160, 408 ảnh test, GPU L4)

| Nhóm | n | Làn đang đi | Làn kề | Không thuộc ta | IoU đường |
|---|---|---|---|---|---|
| Mốc thí điểm 2.826 ảnh, aug mạnh, 320x160 | 5 | 0.7517 ± 0.0061 | 0.5362 ± 0.0068 | 0.4417 ± 0.0029 | 0.8707 ± 0.0014 |
| +1 shard, 320x160 | 5 | 0.7824 ± 0.0018 | 0.5968 ± 0.0066 | 0.4719 ± 0.0018 | 0.8730 ± 0.0010 |
| A: +4 shard, 320x160 (đã có INT8) | 3 | 0.8032 ± 0.0028 | 0.6244 ± 0.0038 | 0.4562 ± 0.0036 | 0.8700 ± 0.0004 |
| HD 640x320 p1s5_reg, thí điểm | 2 | 0.7795 ± 0.0007 | 0.5831 ± 0.0025 | 0.4489 ± 0.0018 | 0.8734 ± 0.0007 |
| Teacher MiT-B2, HD, thí điểm | 1 | 0.8146 | 0.6619